## Reading pkl file instead

In [5]:
import pickle
import csv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from pathlib import Path
from scipy.ndimage import map_coordinates


# ==================================================
# Settings
# ==================================================
PKL_FILE = r"C:\Users\domin\Downloads\processed_data_full_03_25_2025.pkl"

OUT_DIR = Path("BLO_perturbation_keogram_outputs_subtracted")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DX_KM = 1.0
DY_KM = 1.0

DT_SECONDS = 120

FPS = 20
DPI = 200

CMAP = "RdBu_r"

# Make keograms every 5 degrees
KEOGRAM_ANGLES_DEG = np.arange(0, 176, 5)

# Only draw every 15 degrees on the first frame
PLOT_CUT_ANGLES_DEG = np.arange(0, 176, 15)

# Average a wider strip to reduce noise
KEOGRAM_WIDTH_PIXELS = 7

# Keogram display limits
KEOGRAM_TIME_MIN_MAX = 300
KEOGRAM_DISTANCE_MIN = -250
KEOGRAM_DISTANCE_MAX = 250

# Remove the average temporal slope from each keogram
REMOVE_AVERAGE_TIME_SLOPE = True

# Save the original keograms before detrending
SAVE_RAW_KEOGRAMS = False

# Plot the first valid temporal fit for each keogram
SAVE_FIRST_FIT_PLOTS = True


# ==================================================
# Load PKL perturbation array
# ==================================================
with open(PKL_FILE, "rb") as file:
    data = pickle.load(file)

# PKL format:
# data[5] = ["Perturbation Images", perturbation_array]
# data[7] = ["Datetime", [date, start_time]]
wave_cube = np.asarray(data[5][1], dtype=np.float32)
date_start = data[7][1]

# Expected shape: time, y, x
if wave_cube.ndim != 3:
    raise ValueError(
        f"Expected perturbation array with shape (time, y, x), "
        f"got {wave_cube.shape}"
    )

nt, NY, NX = wave_cube.shape

print("Perturbation array shape:", wave_cube.shape)
print("Date/start time:", date_start)


# ==================================================
# Coordinate grid
# ==================================================
x_km = np.arange(NX) * DX_KM
y_km = np.arange(NY) * DY_KM

times_s = np.arange(nt) * DT_SECONDS
times_min = times_s / 60.0


# ==================================================
# Color limits
# ==================================================
vmax = np.nanpercentile(np.abs(wave_cube), 99)

if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0

vmin = -vmax


# ==================================================
# Animation
# ==================================================
fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        x_km.min(),
        x_km.max(),
        y_km.max(),
        y_km.min(),
    ],
)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04,
)
cbar.set_label("Perturbation amplitude")

title = ax.set_title("")

ax.set_xlabel("E-W distance (km)")
ax.set_ylabel("N-S distance (km)")
ax.set_aspect("equal")


def update(frame):
    im.set_data(wave_cube[frame])

    title.set_text(
        f"BLO Perturbation Image | "
        f"Frame {frame}/{nt - 1} | "
        f"t = {times_min[frame]:.1f} min"
    )

    return im, title


ani = FuncAnimation(
    fig,
    update,
    frames=nt,
    interval=1000 / FPS,
    blit=False,
)

out_mp4 = OUT_DIR / "BLO_perturbation_animation.mp4"

writer = FFMpegWriter(
    fps=FPS,
    bitrate=3000,
)

ani.save(
    out_mp4,
    writer=writer,
    dpi=DPI,
)

plt.close(fig)


# ==================================================
# Save first frame with directional keogram cuts
# ==================================================
fig_first, ax_first = plt.subplots(figsize=(7, 6))

im_first = ax_first.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        x_km.min(),
        x_km.max(),
        y_km.max(),
        y_km.min(),
    ],
)

cbar_first = fig_first.colorbar(
    im_first,
    ax=ax_first,
    fraction=0.046,
    pad=0.04,
)
cbar_first.set_label("Perturbation amplitude")

cx_km = 0.5 * (NX - 1) * DX_KM
cy_km = 0.5 * (NY - 1) * DY_KM

half_length_km = 0.5 * min(
    (NX - 1) * DX_KM,
    (NY - 1) * DY_KM,
)

for angle_deg in PLOT_CUT_ANGLES_DEG:
    theta_cut = np.deg2rad(angle_deg)

    dx_line = half_length_km * np.cos(theta_cut)
    dy_line = half_length_km * np.sin(theta_cut)

    x_line = [
        cx_km - dx_line,
        cx_km + dx_line,
    ]

    y_line = [
        cy_km - dy_line,
        cy_km + dy_line,
    ]

    ax_first.plot(
        x_line,
        y_line,
        linewidth=1.4,
        alpha=0.85,
        label=f"{angle_deg:.0f}°",
    )

ax_first.set_title(
    "First Perturbation Frame with Directional Keogram Cuts"
)
ax_first.set_xlabel("E-W distance (km)")
ax_first.set_ylabel("N-S distance (km)")
ax_first.set_aspect("equal")

ax_first.legend(
    loc="upper right",
    fontsize=7,
    ncol=2,
    framealpha=0.8,
)

plt.tight_layout()

out_first = (
    OUT_DIR
    / "first_perturbation_frame_with_keogram_cuts_every_15deg.png"
)

fig_first.savefig(
    out_first,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_first)


# ==================================================
# Directional keogram helper
# ==================================================
def make_directional_keogram(
    wave_cube,
    angle_deg,
    width_pixels=3,
):
    """
    Create a directional keogram through the center
    of the image sequence.

    Parameters
    ----------
    wave_cube : ndarray
        Shape is (time, y, x).

    angle_deg : float
        0 degrees  = E-W slice
        90 degrees = N-S slice

    width_pixels : int
        Number of parallel neighboring slices to average.

    Returns
    -------
    keogram : ndarray
        Shape is (time, distance).

    distance_km : ndarray
        Distance along the cut in km.
    """

    nt_local, ny_local, nx_local = wave_cube.shape

    theta = np.deg2rad(angle_deg)

    # Direction along the keogram cut
    ux = np.cos(theta)
    uy = np.sin(theta)

    # Direction perpendicular to the cut
    px = -np.sin(theta)
    py = np.cos(theta)

    cx = (nx_local - 1) / 2.0
    cy = (ny_local - 1) / 2.0

    half_length_pix = min(nx_local, ny_local) // 2

    distances_pix = np.arange(
        -half_length_pix,
        half_length_pix + 1,
    )

    offsets = (
        np.arange(width_pixels, dtype=float)
        - (width_pixels - 1) / 2.0
    )

    keogram = np.full(
        (nt_local, len(distances_pix)),
        np.nan,
        dtype=float,
    )

    for it in range(nt_local):
        sampled_slices = []

        for off in offsets:
            x_sample = (
                cx
                + distances_pix * ux
                + off * px
            )

            y_sample = (
                cy
                + distances_pix * uy
                + off * py
            )

            valid = (
                (x_sample >= 0)
                & (x_sample <= nx_local - 1)
                & (y_sample >= 0)
                & (y_sample <= ny_local - 1)
            )

            values = np.full(
                len(distances_pix),
                np.nan,
                dtype=float,
            )

            values[valid] = map_coordinates(
                wave_cube[it],
                [
                    y_sample[valid],
                    x_sample[valid],
                ],
                order=1,
                mode="nearest",
            )

            sampled_slices.append(values)

        with np.errstate(invalid="ignore"):
            keogram[it] = np.nanmean(
                sampled_slices,
                axis=0,
            )

    distance_km = distances_pix * DX_KM

    return keogram, distance_km


# ==================================================
# Remove average temporal slope
# ==================================================
def remove_average_temporal_slope(
    keogram,
    time_axis,
):
    """
    Fit a temporal line at each distance column.

    The mean of all fitted slopes is then removed from
    the complete keogram.

    Returns
    -------
    detrended_keogram
    average_slope
    column_slopes
    column_intercepts
    common_trend
    first_valid_column
    first_fit_values
    """

    Z = np.asarray(keogram, dtype=float)
    time_axis = np.asarray(time_axis, dtype=float)

    if Z.ndim != 2:
        raise ValueError(
            f"Expected a 2D keogram, got {Z.shape}"
        )

    if len(time_axis) != Z.shape[0]:
        raise ValueError(
            "The time-axis length does not match the "
            "keogram time dimension."
        )

    ntime, ndistance = Z.shape

    column_slopes = np.full(
        ndistance,
        np.nan,
        dtype=float,
    )

    column_intercepts = np.full(
        ndistance,
        np.nan,
        dtype=float,
    )

    first_valid_column = None
    first_fit_values = None

    # Fit one temporal line at every distance column
    for j in range(ndistance):
        column = Z[:, j]

        valid = (
            np.isfinite(time_axis)
            & np.isfinite(column)
        )

        if np.count_nonzero(valid) < 2:
            continue

        time_valid = time_axis[valid]
        column_valid = column[valid]

        if np.ptp(time_valid) == 0:
            continue

        slope, intercept = np.polyfit(
            time_valid,
            column_valid,
            1,
        )

        column_slopes[j] = slope
        column_intercepts[j] = intercept

        # Store the first valid fit
        if first_valid_column is None:
            first_valid_column = j

            first_fit_values = (
                slope * time_axis
                + intercept
            )

    valid_slopes = np.isfinite(column_slopes)

    if np.any(valid_slopes):
        average_slope = np.nanmean(
            column_slopes[valid_slopes]
        )
    else:
        average_slope = 0.0

    # Center the common trend so that detrending does
    # not change the overall mean level.
    time_reference = np.nanmean(time_axis)

    common_trend = (
        average_slope
        * (time_axis - time_reference)
    )

    detrended_keogram = (
        Z
        - common_trend[:, np.newaxis]
    )

    return (
        detrended_keogram,
        average_slope,
        column_slopes,
        column_intercepts,
        common_trend,
        first_valid_column,
        first_fit_values,
    )


# ==================================================
# Plot first temporal fit
# ==================================================
def plot_first_temporal_fit(
    raw_keogram,
    time_axis,
    distance_axis,
    column_index,
    fit_values,
    angle_deg,
    slope,
    intercept,
    output_file,
):
    """
    Plot the original time series and fitted line for
    the first valid distance column.
    """

    values = raw_keogram[:, column_index]

    valid = (
        np.isfinite(time_axis)
        & np.isfinite(values)
        & np.isfinite(fit_values)
    )

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.plot(
        time_axis[valid],
        values[valid],
        linewidth=1.0,
        label="Original values",
    )

    ax.plot(
        time_axis[valid],
        fit_values[valid],
        linewidth=2.0,
        label="Linear fit",
    )

    ax.set_title(
        f"First Temporal Fit: {angle_deg:.0f}° Keogram\n"
        f"Distance = {distance_axis[column_index]:.1f} km, "
        f"slope = {slope:.6e} amplitude/min"
    )

    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Perturbation amplitude")

    ax.set_xlim(
        time_axis.min(),
        min(
            KEOGRAM_TIME_MIN_MAX,
            time_axis.max(),
        ),
    )

    ax.grid(
        True,
        alpha=0.3,
    )

    ax.legend()

    plt.tight_layout()

    fig.savefig(
        output_file,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(fig)


# ==================================================
# Create and save directional keograms
# ==================================================
saved_keograms = []
saved_first_fit_plots = []

temporal_slope_summary_rows = []

for angle_deg in KEOGRAM_ANGLES_DEG:
    print(
        f"\nProcessing keogram angle "
        f"{angle_deg:.0f} degrees..."
    )

    raw_keogram, distance_km = make_directional_keogram(
        wave_cube,
        angle_deg,
        width_pixels=KEOGRAM_WIDTH_PIXELS,
    )

    # Save original keogram if requested
    if SAVE_RAW_KEOGRAMS:
        raw_npy = (
            OUT_DIR
            / f"BLO_keogram_angle_{angle_deg:03.0f}_raw.npy"
        )

        np.save(
            raw_npy,
            raw_keogram,
        )

    # ----------------------------------------------
    # Remove average temporal slope
    # ----------------------------------------------
    if REMOVE_AVERAGE_TIME_SLOPE:
        (
            keogram,
            average_time_slope,
            column_time_slopes,
            column_time_intercepts,
            removed_time_trend,
            first_valid_column,
            first_fit_values,
        ) = remove_average_temporal_slope(
            raw_keogram,
            times_min,
        )
    else:
        keogram = raw_keogram.copy()

        average_time_slope = 0.0

        column_time_slopes = np.full(
            raw_keogram.shape[1],
            np.nan,
        )

        column_time_intercepts = np.full(
            raw_keogram.shape[1],
            np.nan,
        )

        removed_time_trend = np.zeros(
            raw_keogram.shape[0],
        )

        first_valid_column = None
        first_fit_values = None

    valid_column_slopes = column_time_slopes[
        np.isfinite(column_time_slopes)
    ]

    if valid_column_slopes.size > 0:
        median_time_slope = np.nanmedian(
            valid_column_slopes
        )

        slope_standard_deviation = np.nanstd(
            valid_column_slopes
        )

        number_valid_slopes = valid_column_slopes.size
    else:
        median_time_slope = np.nan
        slope_standard_deviation = np.nan
        number_valid_slopes = 0

    temporal_slope_summary_rows.append({
        "angle_deg":
            float(angle_deg),

        "average_temporal_slope_amplitude_per_min":
            float(average_time_slope),

        "median_temporal_slope_amplitude_per_min":
            float(median_time_slope),

        "temporal_slope_std_amplitude_per_min":
            float(slope_standard_deviation),

        "number_valid_distance_columns":
            int(number_valid_slopes),
    })

    print(
        "  Average temporal slope removed: "
        f"{average_time_slope:.8e} amplitude/min"
    )

    # ----------------------------------------------
    # Plot the first valid fit
    # ----------------------------------------------
    if (
        SAVE_FIRST_FIT_PLOTS
        and first_valid_column is not None
        and first_fit_values is not None
    ):
        first_fit_png = (
            OUT_DIR
            / (
                f"BLO_keogram_angle_"
                f"{angle_deg:03.0f}_first_temporal_fit.png"
            )
        )

        plot_first_temporal_fit(
            raw_keogram=raw_keogram,
            time_axis=times_min,
            distance_axis=distance_km,
            column_index=first_valid_column,
            fit_values=first_fit_values,
            angle_deg=angle_deg,
            slope=column_time_slopes[first_valid_column],
            intercept=column_time_intercepts[first_valid_column],
            output_file=first_fit_png,
        )

        saved_first_fit_plots.append(
            first_fit_png
        )

    # ----------------------------------------------
    # Save detrended keogram plot
    # ----------------------------------------------
    fig_k, ax_k = plt.subplots(
        figsize=(9, 5)
    )

    im_k = ax_k.imshow(
        keogram.T,
        origin="lower",
        aspect="auto",
        cmap=CMAP,
        vmin=vmin,
        vmax=vmax,
        extent=[
            times_min.min(),
            times_min.max(),
            distance_km.min(),
            distance_km.max(),
        ],
    )

    cbar_k = fig_k.colorbar(
        im_k,
        ax=ax_k,
        fraction=0.046,
        pad=0.04,
    )

    cbar_k.set_label(
        "Perturbation amplitude"
    )

    ax_k.set_title(
        f"BLO Directional Keogram: "
        f"{angle_deg:.0f}°\n"
        f"Average temporal slope removed: "
        f"{average_time_slope:.3e} amplitude/min"
    )

    ax_k.set_xlabel("Time (min)")
    ax_k.set_ylabel(
        "Distance along slice (km)"
    )

    ax_k.set_xlim(
        0,
        KEOGRAM_TIME_MIN_MAX,
    )

    ax_k.set_ylim(
        KEOGRAM_DISTANCE_MIN,
        KEOGRAM_DISTANCE_MAX,
    )

    plt.tight_layout()

    out_png = (
        OUT_DIR
        / f"BLO_keogram_angle_{angle_deg:03.0f}.png"
    )

    out_npy = (
        OUT_DIR
        / f"BLO_keogram_angle_{angle_deg:03.0f}.npy"
    )

    fig_k.savefig(
        out_png,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(fig_k)

    # Save detrended keogram
    np.save(
        out_npy,
        keogram,
    )

    saved_keograms.append(
        out_png
    )

    # Save temporal slopes for all distance columns
    np.save(
        OUT_DIR
        / (
            f"BLO_keogram_angle_"
            f"{angle_deg:03.0f}_column_time_slopes.npy"
        ),
        column_time_slopes,
    )

    # Save temporal intercepts for all columns
    np.save(
        OUT_DIR
        / (
            f"BLO_keogram_angle_"
            f"{angle_deg:03.0f}_column_time_intercepts.npy"
        ),
        column_time_intercepts,
    )

    # Save the common trend removed from the keogram
    np.save(
        OUT_DIR
        / (
            f"BLO_keogram_angle_"
            f"{angle_deg:03.0f}_removed_time_trend.npy"
        ),
        removed_time_trend,
    )


# ==================================================
# Save temporal detrending summary CSV
# ==================================================
temporal_slope_csv = (
    OUT_DIR
    / "keogram_average_temporal_slope_summary.csv"
)

with open(
    temporal_slope_csv,
    "w",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "angle_deg",
            "average_temporal_slope_amplitude_per_min",
            "median_temporal_slope_amplitude_per_min",
            "temporal_slope_std_amplitude_per_min",
            "number_valid_distance_columns",
        ],
    )

    writer.writeheader()
    writer.writerows(
        temporal_slope_summary_rows
    )


# ==================================================
# Save arrays and metadata
# ==================================================
np.save(
    OUT_DIR / "times_s.npy",
    times_s,
)

np.save(
    OUT_DIR / "times_min.npy",
    times_min,
)

np.save(
    OUT_DIR / "x_km.npy",
    x_km,
)

np.save(
    OUT_DIR / "y_km.npy",
    y_km,
)

np.save(
    OUT_DIR / "keogram_angles_deg.npy",
    KEOGRAM_ANGLES_DEG,
)

np.save(
    OUT_DIR / "first_frame_cut_angles_deg.npy",
    PLOT_CUT_ANGLES_DEG,
)

with open(
    OUT_DIR / "Date_and_Start_Time.txt",
    "w",
) as file:
    file.write(
        f"Date: {date_start[0]}\n"
    )

    file.write(
        f"Start time: {date_start[1]}\n"
    )

    file.write(
        "Average temporal slope removed from keograms: "
        f"{REMOVE_AVERAGE_TIME_SLOPE}\n"
    )

    file.write(
        "Temporal slope units: "
        "perturbation amplitude per minute\n"
    )

    file.write(
        "The removed trend was centered at the mean time "
        "to preserve the overall keogram mean.\n"
    )


# ==================================================
# Save final frame
# ==================================================
fig_last, ax_last = plt.subplots(
    figsize=(7, 6)
)

im_last = ax_last.imshow(
    wave_cube[-1],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        x_km.min(),
        x_km.max(),
        y_km.max(),
        y_km.min(),
    ],
)

cbar_last = fig_last.colorbar(
    im_last,
    ax=ax_last,
    fraction=0.046,
    pad=0.04,
)

cbar_last.set_label(
    "Perturbation amplitude"
)

ax_last.set_title(
    "Final BLO Perturbation Frame"
)

ax_last.set_xlabel(
    "E-W distance (km)"
)

ax_last.set_ylabel(
    "N-S distance (km)"
)

ax_last.set_aspect("equal")

plt.tight_layout()

out_last = (
    OUT_DIR
    / "final_perturbation_frame.png"
)

fig_last.savefig(
    out_last,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_last)


# ==================================================
# Print outputs
# ==================================================
print("\nSaved:")
print(out_mp4)
print(out_first)
print(out_last)

print("\nDirectional keograms:")
for file_path in saved_keograms:
    print(file_path)

print("\nFirst temporal-fit plots:")
for file_path in saved_first_fit_plots:
    print(file_path)

print("\nTemporal slope summary:")
print(temporal_slope_csv)

print(
    "\nNumber of directional keograms made:",
    len(KEOGRAM_ANGLES_DEG),
)

print(
    "Average temporal slope removal:",
    REMOVE_AVERAGE_TIME_SLOPE,
)

print(
    "FFT analysis was skipped."
)

print(
    "Output folder:",
    OUT_DIR,
)

Perturbation array shape: (280, 1000, 1000)
Date/start time: ['03_25_2025', '20:29:27']

Processing keogram angle 0 degrees...
  Average temporal slope removed: -2.11340471e-03 amplitude/min


C:\Users\domin\AppData\Local\Temp\ipykernel_10884\2243905379.py:365: RuntimeWarning: Mean of empty slice
  keogram[it] = np.nanmean(



Processing keogram angle 5 degrees...
  Average temporal slope removed: -2.06811446e-03 amplitude/min

Processing keogram angle 10 degrees...
  Average temporal slope removed: -2.03252986e-03 amplitude/min

Processing keogram angle 15 degrees...
  Average temporal slope removed: -1.99620413e-03 amplitude/min

Processing keogram angle 20 degrees...
  Average temporal slope removed: -2.07313233e-03 amplitude/min

Processing keogram angle 25 degrees...
  Average temporal slope removed: -2.13045627e-03 amplitude/min

Processing keogram angle 30 degrees...
  Average temporal slope removed: -2.17452084e-03 amplitude/min

Processing keogram angle 35 degrees...
  Average temporal slope removed: -2.20675810e-03 amplitude/min

Processing keogram angle 40 degrees...
  Average temporal slope removed: -2.22912098e-03 amplitude/min

Processing keogram angle 45 degrees...
  Average temporal slope removed: -2.24561409e-03 amplitude/min

Processing keogram angle 50 degrees...
  Average temporal slope 

Date: 03_25_2025
Start time: 20:29:27

## Line plots for BLO keogram

In [15]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ==================================================
# Settings
# ==================================================
OUT_DIR = Path("BLO_perturbation_keogram_outputs_subtracted")

ANGLE_DEG = 40

KEOGRAM_FILE = OUT_DIR / f"BLO_keogram_angle_{ANGLE_DEG:03d}.npy"
TIMES_FILE = OUT_DIR / "times_s.npy"

DPI = 200
CMAP = "RdBu_r"

TIMES_TO_PLOT = [0, 100, 150, 200]

KEOGRAM_TIME_MIN_MAX = 300
KEOGRAM_DISTANCE_MIN = -250
KEOGRAM_DISTANCE_MAX = 250

ROWS_TO_PLOT_KM = [-250, 0, 250]


# ==================================================
# Read keogram
# ==================================================
keogram = np.load(KEOGRAM_FILE)
times_s = np.load(TIMES_FILE)
times_min = times_s / 60.0

nt, nspace = keogram.shape

distance_km = np.arange(-(nspace // 2), nspace // 2 + 1)

if len(distance_km) != nspace:
    distance_km = np.arange(nspace) - nspace // 2


# ==================================================
# Select rows at -250 km, 0 km, and 250 km
# ==================================================
idx_neg250 = np.argmin(np.abs(distance_km - (-250)))
idx_0km = np.argmin(np.abs(distance_km - 0))
idx_pos250 = np.argmin(np.abs(distance_km - 250))

row_neg250 = keogram[:, idx_neg250]
row_0km = keogram[:, idx_0km]
row_pos250 = keogram[:, idx_pos250]


# ==================================================
# Select time columns
# ==================================================
column_indices = []
columns = []

for t in TIMES_TO_PLOT:
    idx = np.argmin(np.abs(times_min - t))
    column_indices.append(idx)
    columns.append(keogram[idx, :])


print("Selected rows:")
print(f"-250 km row index: {idx_neg250}, distance = {distance_km[idx_neg250]} km")
print(f"   0 km row index: {idx_0km}, distance = {distance_km[idx_0km]} km")
print(f" 250 km row index: {idx_pos250}, distance = {distance_km[idx_pos250]} km")

print("\nSelected columns:")
for t, idx in zip(TIMES_TO_PLOT, column_indices):
    print(f"{t:6.1f} min -> frame {idx}, actual time = {times_min[idx]:.2f} min")


# ==================================================
# Plot keogram with selected rows and time columns
# ==================================================
vmax = np.nanpercentile(np.abs(keogram), 99)

if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0

vmin = -vmax

fig, ax = plt.subplots(figsize=(9, 5))

im = ax.imshow(
    keogram.T,
    origin="lower",
    aspect="auto",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        times_min.min(),
        times_min.max(),
        distance_km.min(),
        distance_km.max(),
    ],
)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Perturbation amplitude")

ax.axhline(
    distance_km[idx_neg250],
    color="black",
    linewidth=2.0,
    label="-250 km",
)

ax.axhline(
    distance_km[idx_0km],
    color="gray",
    linewidth=2.0,
    linestyle="--",
    label="0 km",
)

ax.axhline(
    distance_km[idx_pos250],
    color="red",
    linewidth=2.0,
    label="250 km",
)

time_colors = ["black", "blue", "green", "purple"]

for t, idx, color in zip(TIMES_TO_PLOT, column_indices, time_colors):
    ax.axvline(
        times_min[idx],
        color=color,
        linewidth=1.8,
        linestyle=":",
        label=f"{t:.0f} min",
    )

ax.set_xlim(0, KEOGRAM_TIME_MIN_MAX)
ax.set_ylim(KEOGRAM_DISTANCE_MIN, KEOGRAM_DISTANCE_MAX)

ax.set_title(f"{ANGLE_DEG}° BLO Directional Keogram with Selected Rows and Time Columns")
ax.set_xlabel("Time (min)")
ax.set_ylabel("Distance along slice (km)")
ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()

out_keogram_plot = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_with_selected_rows_and_columns.png"
fig.savefig(out_keogram_plot, dpi=DPI, bbox_inches="tight")
plt.close(fig)


# ==================================================
# Plot -250 km row
# ==================================================
fig_neg250, ax_neg250 = plt.subplots(figsize=(8, 4))

ax_neg250.plot(times_min, row_neg250, linewidth=1.5)

ax_neg250.set_xlim(0, KEOGRAM_TIME_MIN_MAX)

ax_neg250.set_title(f"-250 km Row of {ANGLE_DEG}° BLO Keogram")
ax_neg250.set_xlabel("Time (min)")
ax_neg250.set_ylabel("Perturbation amplitude")
ax_neg250.grid(True, alpha=0.3)

plt.tight_layout()

out_neg250 = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_minus250km_row.png"
fig_neg250.savefig(out_neg250, dpi=DPI, bbox_inches="tight")
plt.close(fig_neg250)


# ==================================================
# Plot 0-km row
# ==================================================
fig_0, ax_0 = plt.subplots(figsize=(8, 4))

ax_0.plot(times_min, row_0km, linewidth=1.5)

ax_0.set_xlim(0, KEOGRAM_TIME_MIN_MAX)

ax_0.set_title(f"0-km Row of {ANGLE_DEG}° BLO Keogram")
ax_0.set_xlabel("Time (min)")
ax_0.set_ylabel("Perturbation amplitude")
ax_0.grid(True, alpha=0.3)

plt.tight_layout()

out_0 = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_0km_row.png"
fig_0.savefig(out_0, dpi=DPI, bbox_inches="tight")
plt.close(fig_0)


# ==================================================
# Plot 250 km row
# ==================================================
fig_pos250, ax_pos250 = plt.subplots(figsize=(8, 4))

ax_pos250.plot(times_min, row_pos250, linewidth=1.5)

ax_pos250.set_xlim(0, KEOGRAM_TIME_MIN_MAX)

ax_pos250.set_title(f"250 km Row of {ANGLE_DEG}° BLO Keogram")
ax_pos250.set_xlabel("Time (min)")
ax_pos250.set_ylabel("Perturbation amplitude")
ax_pos250.grid(True, alpha=0.3)

plt.tight_layout()

out_pos250 = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_250km_row.png"
fig_pos250.savefig(out_pos250, dpi=DPI, bbox_inches="tight")
plt.close(fig_pos250)


# ==================================================
# Plot selected time columns/profiles
# ==================================================
fig_c, ax_c = plt.subplots(figsize=(6, 7))

for t, idx, col, color in zip(TIMES_TO_PLOT, column_indices, columns, time_colors):
    ax_c.plot(
        col,
        distance_km,
        linewidth=1.5,
        color=color,
        label=f"{t:.0f} min",
    )

ax_c.axhline(
    distance_km[idx_neg250],
    color="black",
    linestyle="--",
    linewidth=1.2,
    label="-250 km",
)

ax_c.axhline(
    distance_km[idx_0km],
    color="gray",
    linestyle="--",
    linewidth=1.2,
    label="0 km",
)

ax_c.axhline(
    distance_km[idx_pos250],
    color="red",
    linestyle="--",
    linewidth=1.2,
    label="250 km",
)

ax_c.set_ylim(KEOGRAM_DISTANCE_MIN, KEOGRAM_DISTANCE_MAX)

ax_c.set_title(f"Profiles Along {ANGLE_DEG}° Slice at Selected Times")
ax_c.set_xlabel("Perturbation amplitude")
ax_c.set_ylabel("Distance along slice (km)")
ax_c.grid(True, alpha=0.3)
ax_c.legend(loc="best", fontsize=8)

plt.tight_layout()

time_label = "_".join([f"{int(t)}" for t in TIMES_TO_PLOT])
out_columns = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_profiles_{time_label}min.png"
fig_c.savefig(out_columns, dpi=DPI, bbox_inches="tight")
plt.close(fig_c)


# ==================================================
# Print outputs
# ==================================================
print("\nSaved:")
print(out_keogram_plot)
print(out_neg250)
print(out_0)
print(out_pos250)
print(out_columns)

print("\nKeogram shape:")
print(f"time x distance = {keogram.shape}")

print("\nExtracted row arrays:")
print(f"row_neg250 shape = {row_neg250.shape}")
print(f"row_0km shape = {row_0km.shape}")
print(f"row_pos250 shape = {row_pos250.shape}")

print("\nExtracted column arrays:")
for t, col in zip(TIMES_TO_PLOT, columns):
    print(f"{t:.0f} min column shape = {col.shape}")

print("\nPlot limits:")
print(f"Time range: 0 to {KEOGRAM_TIME_MIN_MAX} min")
print(f"Distance range: {KEOGRAM_DISTANCE_MIN} to {KEOGRAM_DISTANCE_MAX} km")

Selected rows:
-250 km row index: 250, distance = -250 km
   0 km row index: 500, distance = 0 km
 250 km row index: 750, distance = 250 km

Selected columns:
   0.0 min -> frame 0, actual time = 0.00 min
 100.0 min -> frame 50, actual time = 100.00 min
 150.0 min -> frame 75, actual time = 150.00 min
 200.0 min -> frame 100, actual time = 200.00 min

Saved:
BLO_perturbation_keogram_outputs_new\BLO_keogram_040_with_selected_rows_and_columns.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_040_minus250km_row.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_040_0km_row.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_040_250km_row.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_040_profiles_0_100_150_200min.png

Keogram shape:
time x distance = (280, 1001)

Extracted row arrays:
row_neg250 shape = (280,)
row_0km shape = (280,)
row_pos250 shape = (280,)

Extracted column arrays:
0 min column shape = (1001,)
100 min column shape = (1001,)
150 min column shape = (1001,)
200

## Overlay
Add another first row figure where each selected vertical profile is converted into an equivalent time span of 25 minutes, then overlaid at its correct center time: 0, 100, 150, and 200 min.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ==================================================
# Settings
# ==================================================

BASE_OUT_DIR = Path("BLO_perturbation_keogram_outputs_subtracted")

ANGLE_DEG = 45

KEOGRAM_TIME_MIN_MAX = 300

# Dynamic distance range
KEOGRAM_DISTANCE_MIN = 0
KEOGRAM_DISTANCE_MAX = 250

PROFILE_TOTAL_TIME_MIN = 15.0

TIMES_TO_PLOT = [0, 50, 100, 125, 150, 175, 200, 250]

DPI = 200
CMAP = "RdBu_r"


# ==================================================
# Helper function
# ==================================================
def distance_filename_label(distance):
    distance = int(round(distance))

    if distance < 0:
        return f"minus{abs(distance)}"
    elif distance > 0:
        return f"plus{distance}"
    else:
        return "0"


range_folder = (
    f"range_"
    f"{distance_filename_label(KEOGRAM_DISTANCE_MIN)}_"
    f"to_"
    f"{distance_filename_label(KEOGRAM_DISTANCE_MAX)}"
)

OUT_DIR = BASE_OUT_DIR / range_folder
OUT_DIR.mkdir(parents=True, exist_ok=True)


KEOGRAM_FILE = (
    BASE_OUT_DIR
    / f"BLO_keogram_angle_{ANGLE_DEG:03d}.npy"
)

TIMES_FILE = (
    BASE_OUT_DIR
    / "times_s.npy"
)


# ==================================================
# Helper function for filename-safe distance labels
# ==================================================
def distance_filename_label(distance):
    """
    Convert a distance into a filename-safe label.

    Examples:
        -100 -> minus100km
           0 -> 0km
         250 -> plus250km
    """
    distance_int = int(round(distance))

    if distance_int < 0:
        return f"minus{abs(distance_int)}km"
    elif distance_int > 0:
        return f"plus{distance_int}km"
    else:
        return "0km"


min_distance_label = distance_filename_label(KEOGRAM_DISTANCE_MIN)
max_distance_label = distance_filename_label(KEOGRAM_DISTANCE_MAX)


# ==================================================
# Read keogram
# ==================================================
keogram = np.load(KEOGRAM_FILE)
times_s = np.load(TIMES_FILE)
times_min = times_s / 60.0

nt, nspace = keogram.shape

distance_km = np.arange(-(nspace // 2), nspace // 2 + 1)

if len(distance_km) != nspace:
    distance_km = np.arange(nspace) - nspace // 2


# ==================================================
# Validate requested distance range
# ==================================================
available_distance_min = np.nanmin(distance_km)
available_distance_max = np.nanmax(distance_km)

if KEOGRAM_DISTANCE_MIN >= KEOGRAM_DISTANCE_MAX:
    raise ValueError(
        "KEOGRAM_DISTANCE_MIN must be smaller than "
        "KEOGRAM_DISTANCE_MAX."
    )

if KEOGRAM_DISTANCE_MIN < available_distance_min:
    raise ValueError(
        f"Requested minimum distance is {KEOGRAM_DISTANCE_MIN} km, "
        f"but the keogram only extends to {available_distance_min} km."
    )

if KEOGRAM_DISTANCE_MAX > available_distance_max:
    raise ValueError(
        f"Requested maximum distance is {KEOGRAM_DISTANCE_MAX} km, "
        f"but the keogram only extends to {available_distance_max} km."
    )


# ==================================================
# Select rows at minimum distance, 0 km,
# and maximum distance
# ==================================================
idx_min = np.argmin(
    np.abs(distance_km - KEOGRAM_DISTANCE_MIN)
)

idx_0km = np.argmin(
    np.abs(distance_km - 0)
)

idx_max = np.argmin(
    np.abs(distance_km - KEOGRAM_DISTANCE_MAX)
)

actual_distance_min = distance_km[idx_min]
actual_distance_0 = distance_km[idx_0km]
actual_distance_max = distance_km[idx_max]

row_min = keogram[:, idx_min]
row_0km = keogram[:, idx_0km]
row_max = keogram[:, idx_max]


# ==================================================
# Select time columns
# ==================================================
column_indices = []
columns = []

for requested_time in TIMES_TO_PLOT:
    idx = np.argmin(
        np.abs(times_min - requested_time)
    )

    column_indices.append(idx)
    columns.append(keogram[idx, :])


# ==================================================
# Print selected rows and columns
# ==================================================
print("Selected rows:")

print(
    f"{KEOGRAM_DISTANCE_MIN:7.1f} km requested -> "
    f"row index {idx_min}, "
    f"actual distance = {actual_distance_min:.1f} km"
)

print(
    f"{0:7.1f} km requested -> "
    f"row index {idx_0km}, "
    f"actual distance = {actual_distance_0:.1f} km"
)

print(
    f"{KEOGRAM_DISTANCE_MAX:7.1f} km requested -> "
    f"row index {idx_max}, "
    f"actual distance = {actual_distance_max:.1f} km"
)

print("\nSelected columns:")

for requested_time, idx in zip(
    TIMES_TO_PLOT,
    column_indices,
):
    print(
        f"{requested_time:6.1f} min -> "
        f"frame {idx}, "
        f"actual time = {times_min[idx]:.2f} min"
    )


# ==================================================
# Color limits
# ==================================================
vmax = np.nanpercentile(
    np.abs(keogram),
    99,
)

if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0

vmin = -vmax

time_colors = [
    "black",
    "blue",
    "green",
    "purple",
    "orange",
    "pink",
    "brown",
    "magenta",
]

if len(TIMES_TO_PLOT) > len(time_colors):
    raise ValueError(
        "There are more TIMES_TO_PLOT entries than colors "
        "in time_colors."
    )


# ==================================================
# Plot keogram with selected rows and time columns
# ==================================================
fig, ax = plt.subplots(
    figsize=(9, 5)
)

im = ax.imshow(
    keogram.T,
    origin="lower",
    aspect="auto",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        times_min.min(),
        times_min.max(),
        distance_km.min(),
        distance_km.max(),
    ],
)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04,
)

cbar.set_label(
    "Perturbation amplitude"
)

ax.axhline(
    actual_distance_min,
    color="black",
    linewidth=2.0,
    label=f"{actual_distance_min:.0f} km",
)

ax.axhline(
    actual_distance_0,
    color="gray",
    linewidth=2.0,
    linestyle="--",
    label=f"{actual_distance_0:.0f} km",
)

ax.axhline(
    actual_distance_max,
    color="red",
    linewidth=2.0,
    label=f"{actual_distance_max:.0f} km",
)

for requested_time, idx, color in zip(
    TIMES_TO_PLOT,
    column_indices,
    time_colors,
):
    ax.axvline(
        times_min[idx],
        color=color,
        linewidth=1.8,
        linestyle=":",
        label=f"{requested_time:.0f} min",
    )

ax.set_xlim(
    0,
    KEOGRAM_TIME_MIN_MAX,
)

ax.set_ylim(
    KEOGRAM_DISTANCE_MIN,
    KEOGRAM_DISTANCE_MAX,
)

ax.set_title(
    f"{ANGLE_DEG}° BLO Directional Keogram "
    f"with Selected Rows and Time Columns"
)

ax.set_xlabel(
    "Time (min)"
)

ax.set_ylabel(
    "Distance along slice (km)"
)

ax.legend(
    loc="upper right",
    fontsize=8,
)

plt.tight_layout()

out_keogram_plot = (
    OUT_DIR
    / (
        f"BLO_keogram_{ANGLE_DEG:03d}_"
        f"range_{min_distance_label}_to_{max_distance_label}_"
        f"with_selected_rows_and_columns.png"
    )
)

fig.savefig(
    out_keogram_plot,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig)


# ==================================================
# Plot minimum-distance row
# ==================================================
fig_min, ax_min = plt.subplots(
    figsize=(8, 4)
)

ax_min.plot(
    times_min,
    row_min,
    linewidth=1.5,
)

ax_min.set_xlim(
    0,
    KEOGRAM_TIME_MIN_MAX,
)

ax_min.set_title(
    f"{actual_distance_min:.0f} km Row of "
    f"{ANGLE_DEG}° BLO Keogram"
)

ax_min.set_xlabel(
    "Time (min)"
)

ax_min.set_ylabel(
    "Perturbation amplitude"
)

ax_min.grid(
    True,
    alpha=0.3,
)

plt.tight_layout()

out_min = (
    OUT_DIR
    / (
        f"BLO_keogram_{ANGLE_DEG:03d}_"
        f"{min_distance_label}_row.png"
    )
)

fig_min.savefig(
    out_min,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_min)


# ==================================================
# Overlay selected profiles onto minimum-distance row
# ==================================================
distance_mask = (
    (distance_km >= KEOGRAM_DISTANCE_MIN)
    & (distance_km <= KEOGRAM_DISTANCE_MAX)
)

distance_for_overlay = distance_km[distance_mask]

# Normalize the selected distance range:
#
# KEOGRAM_DISTANCE_MIN -> 0
# KEOGRAM_DISTANCE_MAX -> 1
#
# This works for asymmetric ranges such as -100 to 250 km.
distance_norm = (
    distance_for_overlay
    - KEOGRAM_DISTANCE_MIN
) / (
    KEOGRAM_DISTANCE_MAX
    - KEOGRAM_DISTANCE_MIN
)

# Calculate the time offset corresponding to 0 km.
zero_distance_fraction = (
    0.0 - KEOGRAM_DISTANCE_MIN
) / (
    KEOGRAM_DISTANCE_MAX
    - KEOGRAM_DISTANCE_MIN
)

zero_distance_time_offset = (
    zero_distance_fraction
    * PROFILE_TOTAL_TIME_MIN
)

fig_min_overlay, ax_min_overlay = plt.subplots(
    figsize=(9, 4)
)

ax_min_overlay.plot(
    times_min,
    row_min,
    color="black",
    linewidth=1.5,
    label=f"{actual_distance_min:.0f} km row",
)

for requested_time, idx, col, color in zip(
    TIMES_TO_PLOT,
    column_indices,
    columns,
    time_colors,
):
    profile_time = (
        times_min[idx]
        + distance_norm
        * PROFILE_TOTAL_TIME_MIN
    )

    profile_amp = col[distance_mask]

    ax_min_overlay.plot(
        profile_time,
        profile_amp,
        color=color,
        linewidth=1.5,
        alpha=0.8,
        label=(
            f"profile starting at "
            f"{requested_time:.0f} min"
        ),
    )

    ax_min_overlay.axvline(
        times_min[idx],
        color=color,
        linestyle=":",
        linewidth=1.2,
        alpha=0.8,
    )

ax_min_overlay.set_xlim(
    0,
    KEOGRAM_TIME_MIN_MAX,
)

ax_min_overlay.set_title(
    f"{actual_distance_min:.0f} km Row of "
    f"{ANGLE_DEG}° Keogram with Selected Profiles Overlaid\n"
    f"Profiles Start at Selected Times; "
    f"{KEOGRAM_DISTANCE_MIN:.0f} to "
    f"{KEOGRAM_DISTANCE_MAX:.0f} km Mapped to "
    f"{PROFILE_TOTAL_TIME_MIN:.0f} min"
)

ax_min_overlay.set_xlabel(
    "Time (min)"
)

ax_min_overlay.set_ylabel(
    "Perturbation amplitude"
)

ax_min_overlay.grid(
    True,
    alpha=0.3,
)

ax_min_overlay.legend(
    loc="best",
    fontsize=8,
)

plt.tight_layout()

out_min_overlay = (
    OUT_DIR
    / (
        f"BLO_keogram_{ANGLE_DEG:03d}_"
        f"{min_distance_label}_row_with_profiles_"
        f"range_{min_distance_label}_to_{max_distance_label}_"
        f"{PROFILE_TOTAL_TIME_MIN:.0f}min_mapping.png"
    )
)

fig_min_overlay.savefig(
    out_min_overlay,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_min_overlay)


# ==================================================
# Plot 0-km row
# ==================================================
fig_0, ax_0 = plt.subplots(
    figsize=(8, 4)
)

ax_0.plot(
    times_min,
    row_0km,
    linewidth=1.5,
)

ax_0.set_xlim(
    0,
    KEOGRAM_TIME_MIN_MAX,
)

ax_0.set_title(
    f"{actual_distance_0:.0f} km Row of "
    f"{ANGLE_DEG}° BLO Keogram"
)

ax_0.set_xlabel(
    "Time (min)"
)

ax_0.set_ylabel(
    "Perturbation amplitude"
)

ax_0.grid(
    True,
    alpha=0.3,
)

plt.tight_layout()

out_0 = (
    OUT_DIR
    / (
        f"BLO_keogram_{ANGLE_DEG:03d}_"
        f"0km_row.png"
    )
)

fig_0.savefig(
    out_0,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_0)


# ==================================================
# Plot maximum-distance row
# ==================================================
fig_max, ax_max = plt.subplots(
    figsize=(8, 4)
)

ax_max.plot(
    times_min,
    row_max,
    linewidth=1.5,
)

ax_max.set_xlim(
    0,
    KEOGRAM_TIME_MIN_MAX,
)

ax_max.set_title(
    f"{actual_distance_max:.0f} km Row of "
    f"{ANGLE_DEG}° BLO Keogram"
)

ax_max.set_xlabel(
    "Time (min)"
)

ax_max.set_ylabel(
    "Perturbation amplitude"
)

ax_max.grid(
    True,
    alpha=0.3,
)

plt.tight_layout()

out_max = (
    OUT_DIR
    / (
        f"BLO_keogram_{ANGLE_DEG:03d}_"
        f"{max_distance_label}_row.png"
    )
)

fig_max.savefig(
    out_max,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_max)


# ==================================================
# Plot selected time columns/profiles
# ==================================================
fig_c, ax_c = plt.subplots(
    figsize=(6, 7)
)

for requested_time, idx, col, color in zip(
    TIMES_TO_PLOT,
    column_indices,
    columns,
    time_colors,
):
    ax_c.plot(
        col,
        distance_km,
        linewidth=1.5,
        color=color,
        label=f"{requested_time:.0f} min",
    )

ax_c.axhline(
    actual_distance_min,
    color="black",
    linestyle="--",
    linewidth=1.2,
    label=f"{actual_distance_min:.0f} km",
)

ax_c.axhline(
    actual_distance_0,
    color="gray",
    linestyle="--",
    linewidth=1.2,
    label=f"{actual_distance_0:.0f} km",
)

ax_c.axhline(
    actual_distance_max,
    color="red",
    linestyle="--",
    linewidth=1.2,
    label=f"{actual_distance_max:.0f} km",
)

ax_c.set_ylim(
    KEOGRAM_DISTANCE_MIN,
    KEOGRAM_DISTANCE_MAX,
)

ax_c.set_title(
    f"Profiles Along {ANGLE_DEG}° Slice "
    f"at Selected Times"
)

ax_c.set_xlabel(
    "Perturbation amplitude"
)

ax_c.set_ylabel(
    "Distance along slice (km)"
)

ax_c.grid(
    True,
    alpha=0.3,
)

ax_c.legend(
    loc="best",
    fontsize=8,
)

plt.tight_layout()

time_label = "_".join(
    [f"{int(t)}" for t in TIMES_TO_PLOT]
)

out_columns = (
    OUT_DIR
    / (
        f"BLO_keogram_{ANGLE_DEG:03d}_"
        f"profiles_{time_label}min_"
        f"range_{min_distance_label}_to_{max_distance_label}.png"
    )
)

fig_c.savefig(
    out_columns,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_c)


# ==================================================
# Print outputs
# ==================================================
print("\nSaved:")
print(out_keogram_plot)
print(out_min)
print(out_min_overlay)
print(out_0)
print(out_max)
print(out_columns)

print("\nKeogram shape:")
print(
    f"time x distance = {keogram.shape}"
)

print("\nExtracted row arrays:")
print(
    f"row_min shape = {row_min.shape}"
)

print(
    f"row_0km shape = {row_0km.shape}"
)

print(
    f"row_max shape = {row_max.shape}"
)

print("\nExtracted column arrays:")

for requested_time, col in zip(
    TIMES_TO_PLOT,
    columns,
):
    print(
        f"{requested_time:.0f} min "
        f"column shape = {col.shape}"
    )

print("\nPlot limits:")

print(
    f"Time range: 0 to "
    f"{KEOGRAM_TIME_MIN_MAX} min"
)

print(
    f"Distance range: "
    f"{KEOGRAM_DISTANCE_MIN} to "
    f"{KEOGRAM_DISTANCE_MAX} km"
)

print("\nOverlay mapping:")

print(
    f"{KEOGRAM_DISTANCE_MIN:.0f} km "
    f"-> selected time"
)

print(
    f"0 km -> selected time + "
    f"{zero_distance_time_offset:.2f} min"
)

print(
    f"{KEOGRAM_DISTANCE_MAX:.0f} km "
    f"-> selected time + "
    f"{PROFILE_TOTAL_TIME_MIN:.2f} min"
)

Selected rows:
    0.0 km requested -> row index 500, actual distance = 0.0 km
    0.0 km requested -> row index 500, actual distance = 0.0 km
  250.0 km requested -> row index 750, actual distance = 250.0 km

Selected columns:
   0.0 min -> frame 0, actual time = 0.00 min
  50.0 min -> frame 25, actual time = 50.00 min
 100.0 min -> frame 50, actual time = 100.00 min
 125.0 min -> frame 62, actual time = 124.00 min
 150.0 min -> frame 75, actual time = 150.00 min
 175.0 min -> frame 87, actual time = 174.00 min
 200.0 min -> frame 100, actual time = 200.00 min
 250.0 min -> frame 125, actual time = 250.00 min

Saved:
BLO_perturbation_keogram_outputs_subtracted\range_0_to_plus250\BLO_keogram_045_range_0km_to_plus250km_with_selected_rows_and_columns.png
BLO_perturbation_keogram_outputs_subtracted\range_0_to_plus250\BLO_keogram_045_0km_row.png
BLO_perturbation_keogram_outputs_subtracted\range_0_to_plus250\BLO_keogram_045_0km_row_with_profiles_range_0km_to_plus250km_15min_mapping.png
BLO_

### Calculate speed: 
650 km in 1500 s => 433 m/s

650 km is 25/120 of wavelength. => wavelength = 650 km /(25/120) = 3120 km. 